# Chapter 16. Testing and Debugging Your Programs

Companion notebook for *Python from Zero*. Run the cells from top to
bottom with **Shift + Enter**. Some cells stop with an `AssertionError` on
purpose: that is a test catching a bug. Read the message, then keep going.
The pytest runs use the files in `../programs/ch16_tests/`.

## 16.2 Checking with assert

In [1]:
assert 2 + 2 == 4
print("Still running")

Still running


In [2]:
assert 2 + 2 == 5

AssertionError: 

In [3]:
kennels = 20
dogs_in = 22
assert dogs_in <= kennels, f"Only {kennels} kennels but {dogs_in} dogs"

AssertionError: Only 20 kennels but 22 dogs

## 16.3 From checking by hand to test functions

Here is a version of `adoption_fee()` with a bug hiding in it.

In [4]:
BASE_FEES = {"Dog": 150, "Cat": 90, "Rabbit": 45, "Guinea pig": 25}


def adoption_fee(species, age_years, status):
    """Return the adoption fee in dollars for one animal."""
    if status == "Medical hold":
        raise ValueError("Animals on medical hold cannot be adopted")
    fee = BASE_FEES[species]
    if species in ("Dog", "Cat") and age_years > 8:
        fee = fee // 2
    return fee

In [5]:
print(adoption_fee("Dog", 3, "Available"))
print(adoption_fee("Dog", 10, "Available"))
print(adoption_fee("Cat", 2, "Available"))

150
75
90


In [6]:
def test_basic_fees():
    assert adoption_fee("Dog", 3, "Available") == 150
    assert adoption_fee("Rabbit", 3, "Available") == 45


def test_senior_dog_pays_half():
    assert adoption_fee("Dog", 10, "Available") == 75


def test_exactly_eight_counts_as_senior():
    assert adoption_fee("Cat", 8, "Available") == 45


test_basic_fees()
test_senior_dog_pays_half()
test_exactly_eight_counts_as_senior()
print("All tests passed")

AssertionError: 

Fix the bug: an 8-year-old **is** a senior, so `>` must become `>=`.

In [7]:
BASE_FEES = {"Dog": 150, "Cat": 90, "Rabbit": 45, "Guinea pig": 25}


def adoption_fee(species, age_years, status):
    """Return the adoption fee in dollars for one animal."""
    if status == "Medical hold":
        raise ValueError("Animals on medical hold cannot be adopted")
    fee = BASE_FEES[species]
    if species in ("Dog", "Cat") and age_years >= 8:
        fee = fee // 2
    return fee

In [8]:
test_basic_fees()
test_senior_dog_pays_half()
test_exactly_eight_counts_as_senior()
print("All tests passed")

All tests passed


## 16.4 Running pytest from the notebook

In the chapter you run pytest in a terminal. `pytest.main()` runs the same tests from inside Jupyter. `-q` means *quiet*.

In [9]:
import pytest

pytest.main(["-q", "../programs/ch16_tests"])

.

.

.

.

.

...

.

.

                                                    [100%]


21 passed in 1.14s


<ExitCode.OK: 0>

## 16.5 Edge cases

The tested rules, the same code as `programs/ch16_tests/shelter_rules.py`:

In [10]:
BASE_FEES = {"Dog": 150, "Cat": 90, "Rabbit": 45, "Guinea pig": 25}
LATER_YEARS = {"Dog": 5, "Cat": 4}


def clean_species(species):
    """Return species with tidy capitals, so "dog " becomes "Dog"."""
    return species.strip().capitalize()


def human_years(species, age_years):
    """Return a dog's or cat's age in human years."""
    species = clean_species(species)
    if species not in LATER_YEARS:
        raise ValueError(f"No human-years rule for {species}")
    if age_years < 0:
        raise ValueError("Age cannot be negative")
    if age_years < 1:
        return round(age_years * 15, 1)
    if age_years < 2:
        return round(15 + (age_years - 1) * 9, 1)
    return round(24 + (age_years - 2) * LATER_YEARS[species], 1)


def adoption_fee(species, age_years, status):
    """Return the adoption fee in dollars for one animal."""
    species = clean_species(species)
    if species not in BASE_FEES:
        raise ValueError(f"Unknown species: {species}")
    if status == "Medical hold":
        raise ValueError("Animals on medical hold cannot be adopted")
    fee = BASE_FEES[species]
    if species in ("Dog", "Cat") and age_years >= 8:
        fee = fee // 2
    return fee


def food_grams(species, age_years, weight_kg):
    """Return grams of dry food (or pellets) per day for one animal."""
    species = clean_species(species)
    if species == "Dog":
        return round(weight_kg * 15)
    if species == "Cat":
        if age_years < 1:
            return 80
        return 60
    if species == "Rabbit":
        return 80
    if species == "Guinea pig":
        return 40
    raise ValueError(f"Unknown species: {species}")

In [11]:
print(human_years("Dog", 0))
print(adoption_fee("dog ", 3, "Foster"))

0
150


In [12]:
adoption_fee("Dog", 3, "Medical hold")

ValueError: Animals on medical hold cannot be adopted

In [13]:
human_years("Dog", "3")

TypeError: '<' not supported between instances of 'str' and 'int'

In [14]:
print(0.1 + 0.2)
assert 0.1 + 0.2 == 0.3

0.30000000000000004


AssertionError: 

In [15]:
assert 0.1 + 0.2 == pytest.approx(0.3)
print("Close enough")

Close enough


## 16.6 The donations report, as Rosa found it

In [16]:
import csv


def load_donations(path):
    """Read the donations CSV into a list of dicts (amounts as ints)."""
    donations = []
    with open(path, encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            row["amount"] = int(row["amount"])
            donations.append(row)
    return donations


def total(donations):
    """Add up every donation."""
    result = 0
    for gift in donations:
        result += gift["amount"]
    return result


def total_by_month(donations):
    """Return a dict like {"2026-01": 1234, ...}."""
    totals = {}
    for gift in donations:
        month = gift["date"][:7]
        totals[month] = gift["amount"]
    return totals


def average_gift(donations):
    """Return the average donation in dollars."""
    return total(donations) / len(donations)


def print_report(donations):
    print("Sunny Paws donations report")
    print("=" * 27)
    for month, amount in total_by_month(donations).items():
        print(f"{month}  ${amount:>6,}")
    print(f"Total    ${total(donations):>6,}")
    print(f"Average gift: ${average_gift(donations):.2f}")

In [17]:
print_report(load_donations("../data/donations.csv"))

Sunny Paws donations report
2026-01  $    10
2026-02  $   200
2026-03  $    20
2026-04  $    50
2026-05  $   200
2026-06  $    20
Total    $ 7,805
Average gift: $60.04


In [18]:
GIFTS = [
    {"date": "2026-01-03", "donor": "Chris Doyle", "amount": 50},
    {"date": "2026-01-10", "donor": "Maple Falls Bakery", "amount": 100},
    {"date": "2026-02-14", "donor": "Grace Kim", "amount": 25},
]


def test_two_gifts_in_january_are_added():
    monthly = total_by_month(GIFTS)
    assert monthly["2026-01"] == 150


test_two_gifts_in_january_are_added()
print("Passed")

AssertionError: 

## 16.8 The fix

In [19]:
def total_by_month(donations):
    """Return a dict like {"2026-01": 1234, ...}."""
    totals = {}
    for gift in donations:
        month = gift["date"][:7]
        totals[month] = totals.get(month, 0) + gift["amount"]
    return totals


def average_gift(donations):
    """Return the average donation in dollars (0 if there are none)."""
    if len(donations) == 0:
        return 0
    return total(donations) / len(donations)


test_two_gifts_in_january_are_added()
print("Passed")
print(average_gift([]))

Passed
0


In [20]:
print_report(load_donations("../data/donations.csv"))

Sunny Paws donations report
2026-01  $ 1,855
2026-02  $ 1,050
2026-03  $ 1,135
2026-04  $ 1,425
2026-05  $ 1,410
2026-06  $   930
Total    $ 7,805
Average gift: $60.04


## 16.12 Exercises

1. Write two `assert` statements that check `food_grams()` for Peanut
   (a 4.5-year-old, 28.6 kg dog: 429 g) and for a 0.5-year-old kitten
   (80 g). Run them.
2. Add a test to `test_shelter_rules.py` checking that a 1-year-old dog
   is 15 human years and a 2-year-old cat is 24. Run pytest.
3. Write tests for two edge cases: `food_grams("Hamster", 1, 0.1)` raises
   `ValueError`, and a 10-year-old guinea pig still pays the full $25.
4. Write a function `is_senior(species, age_years)` and three tests for it.
5. Test first: write tests for `total_by_method(donations)` (totals per
   payment method), watch them fail, then write the function.
6. Find and fix the planted bug in `count_available(animals)` (see the
   chapter) by writing a test that fails first.
7. Use the debugger (pdb, IDLE or VS Code) to step through the buggy
   `top_donor(donations)` from the chapter and fix it.
8. Write a smoke test that loads `../../data/donations.csv` and checks the
   total is 7,805 and that the months add up to the total.